# Tests with Spanish sustainability problem

In [1]:
from desdeo.problem.testproblems.spanish_sustainability_problem import spanish_sustainability_problem
from desdeo.mcdm.nimbus import solve_sub_problems
from desdeo.tools.generics import SolverResults

problem = spanish_sustainability_problem()



print(f"Ideal values: {problem.get_ideal_point()}")
print(f"Nadir values: {problem.get_nadir_point()} (approximations!)")

Ideal values: {'f_1': 1.17, 'f_2': 1.98, 'f_3': 2.93}
Nadir values: {'f_1': 1.15, 'f_2': 0.63, 'f_3': 1.52} (approximations!)


We can then update our problem with the new ideal and nadir point values:

Iteration 1

In [2]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions
from desdeo.tools import PyomoIpoptSolver, IpoptOptions, ScipyMinimizeSolver, GurobipySolver, PyomoGurobiSolver, ProximalSolver, PyomoBonminSolver, find_compatible_solvers

solver = PyomoIpoptSolver

X = [7.0576,89.6530,16.5170,2.0723,1.0000,1.9501,17.2040,13.3250,70.0000,104.1900,119.9700]

# Map the full X array to the variable (respects variable bounds)
initial_values = {"X": X}

# Set solver options with initial values
solver_options = IpoptOptions()
solver_options.max_iter = 10000000
solver_options.tol = 1e-8
print(f"Compatible solvers: {find_compatible_solvers(problem)}")

reference_point = {"f_1": 1.15, "f_2": 1, "f_3": 2.9}

print (f"Problem variables and their bounds:")
print (problem.variables)


for var in problem.variables:
    print(f"{var.name} with bounds {var.get_lowerbound_values()} and {var.get_upperbound_values()}")

Compatible solvers: [<class 'desdeo.tools.pyomo_solver_interfaces.PyomoIpoptSolver'>]
Problem variables and their bounds:
[TensorVariable(name="Variables 'x_1' through 'x_11' defined as a vector.", symbol='X', variable_type=<VariableTypeEnum.real: 'real'>, shape=[11], lowerbounds=['List', 1.0, 60.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 40.0, 75.0, 80.0], upperbounds=['List', 40.0, 90.0, 25.0, 3.0, 40.0, 15.0, 30.0, 25.0, 70.0, 105.0, 120.0], initial_values=['List', 7.0576, 89.653, 16.517, 2.0723, 1.0, 1.9501, 17.204, 13.325, 70.0, 104.19, 119.97])]
Variables 'x_1' through 'x_11' defined as a vector. with bounds [1.0, 60.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 40.0, 75.0, 80.0] and [40.0, 90.0, 25.0, 3.0, 40.0, 15.0, 30.0, 25.0, 70.0, 105.0, 120.0]


In [3]:
#results = rpm_solve_solutions(problem, reference_point=reference_point, solver=solver, solver_options=solver_options)

results = rpm_solve_solutions(
    problem, 
    reference_point=reference_point, 
    solver=solver, 
    solver_options=solver_options,
)
for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print(f"Constraint duals \t\t\t {result.lagrange_multipliers}")
    print("---")

Solution 1:
Objective function values 		 {'f_1': 1.166081403054287, 'f_2': 0.8642413179106124, 'f_3': 2.7582076045077497}
Decision variable values 		 {'X': [6.434360381909828, 89.99999981942952, 16.51716226081498, 2.0723324295925982, 1.0000000165485725, 1.9440620361097365, 21.23483694091475, 13.348911173877653, 70.0, 104.99999886538495, 80.00000273433609], '_alpha': 0.10056190682622686}
Constraint values 			 {'con_1': -0.007812599069438875, 'con_2': -191844.15528573375, 'con_3': -0.5188933936921925, 'con_4': -0.48773179994649496, 'con_5': -0.8215169295925981, 'con_6': 2.959259814971915e-08, 'con_7': -3.3580723446391403e-07, 'con_8': -0.12320388886388844, 'con_9': -0.09259542655159181, 'con_10': -1.1873205309411876, 'con_11': -0.2945513290856818, 'con_12': -2.789573227771222, 'con_13': -1179.2159111796723, 'con_14': -0.00013240259151636735, 'con_15': -5053.2051136313075, 'con_16': -3732.703275212454, 'con_17': -10.984282707354733, 'con_18': -7.17161722285859, 'con_19': -1.99410375706342

# Example with other problems

In [4]:
from desdeo.problem.testproblems.binh_and_korn_problem import binh_and_korn
from desdeo.problem.testproblems.cake_problem import best_cake_problem
from desdeo.problem.testproblems.forest_problem import forest_problem
from desdeo.problem.testproblems.rocket_injector_design_problem import rocket_injector_design
from desdeo.problem.testproblems.knapsack_problem import simple_knapsack

problems = [
    binh_and_korn(),
    best_cake_problem(),
    rocket_injector_design(),
    simple_knapsack(),
]

for problem in problems:
    print("Problem name: ", problem.name)
    print("Available solvers:")
    solvers = find_compatible_solvers(problem)
    #Check if solvers contain PyomoIpoptSolver
    print(solvers)
    print("Objectives", len(problem.objectives))
    print("---")

problem = rocket_injector_design()

for obj in problem.objectives:
    print(f"Objective {obj.name} with symbol {obj.symbol}")

reference_point = {"TF_max": 2, "TT_max": 1, "Xcc_max": 2.9}


results = rpm_solve_solutions(
    problem, 
    reference_point=reference_point, 
    solver=solver, 
    solver_options=solver_options,
)
for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print(f"Constraint duals \t\t\t {result.lagrange_multipliers}")
    print("---")


Problem name:  The Binh and Korn function
Available solvers:
[<class 'desdeo.tools.pyomo_solver_interfaces.PyomoIpoptSolver'>, <class 'desdeo.tools.ng_solver_interfaces.NevergradGenericSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyMinimizeSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyDeSolver'>]
Objectives 2
---
Problem name:  Cake problem
Available solvers:
[<class 'desdeo.tools.pyomo_solver_interfaces.PyomoIpoptSolver'>, <class 'desdeo.tools.ng_solver_interfaces.NevergradGenericSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyMinimizeSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyDeSolver'>]
Objectives 5
---
Problem name:  Rocket Injector Design Problem
Available solvers:
[<class 'desdeo.tools.pyomo_solver_interfaces.PyomoIpoptSolver'>, <class 'desdeo.tools.ng_solver_interfaces.NevergradGenericSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyMinimizeSolver'>, <class 'desdeo.tools.scipy_solver_interfaces.ScipyDeSolver'>]
O

# Utopia Problem

In [5]:
import json
import os
from desdeo.problem import Problem
from desdeo.tools.utils import guess_best_solver

def load_problem_from_json(json_file_path: str) -> Problem:
    """Load a problem from a JSON file."""
    with open(json_file_path, "r") as f:
        data = json.load(f)
    return Problem.model_validate(data, by_name=True)

utopia_problem=load_problem_from_json("C:\\Users\\Giomara\\Documents\\Projects\\DESDEO_webui\\DESDEO\\datasets\\problemModels\\utopia_forest.json")

solvers = find_compatible_solvers(utopia_problem)

recommended_solver = guess_best_solver(utopia_problem)

#Check if solvers contain PyomoIpoptSolver
print(recommended_solver)

for obj in utopia_problem.objectives:
    print(f"Objective {obj.name} with symbol {obj.symbol}")

reference_point = {"net_present_value": 2, "timber_volume": 1, "logging_revenue": 2.9, "stored_co2": 1.5}

results = rpm_solve_solutions(
    utopia_problem, 
    reference_point=reference_point, 
    solver=recommended_solver, 
    solver_options=None,
)
for i, result in enumerate(results):
    print(f"Solution {i+1}:")
    print(f"Objective function values \t\t {result.optimal_objectives}")
    print(f"Decision variable values \t\t {result.optimal_variables}")
    print(f"Constraint values \t\t\t {result.constraint_values}")
    print(f"Constraint duals \t\t\t {result.lagrange_multipliers}")
    print("---")

print(utopia_problem.is_twice_differentiable)


<class 'desdeo.tools.gurobipy_solver_interfaces.GurobipySolver'>
Objective Net present value / â‚¬ with symbol net_present_value
Objective Final timber volume / m^3
(start 3001m^3) with symbol timber_volume
Objective Logging revenue / â‚¬ with symbol logging_revenue
Objective Stored CO2 / (yearÂ·tonne) with symbol stored_co2
Set parameter Username
Set parameter LicenseID to value 2703269
Academic license - for non-commercial use only - expires 2026-09-04
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10850H CPU @ 2.70GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 40 rows, 1017 columns and 3205 nonzeros
Model fingerprint: 0x3a6181e1
Variable types: 9 continuous, 1008 integer (1008 binary)
Coefficient statistics:
  Matrix range     [1e-06, 8e+04]
  Objective range  [3e-12, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range      